# 01 — Exploratory Data Analysis

**Dataset:** Give Me Some Credit (Kaggle)  
**Goal:** Understand the data distribution, identify missing values, and document the class imbalance before building the preprocessing pipeline.

Key questions:
1. What is the class balance? (target: `SeriousDlqin2yrs`)
2. Which features have missing values, and how many?
3. What are the distributions of each feature, and are there extreme outliers?
4. Are any features correlated with each other or the target?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = '../data/cs-training.csv'

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)

COLUMN_MAP = {
    'RevolvingUtilizationOfUnsecuredLines': 'revolving_utilization',
    'age': 'age',
    'NumberOfTime30-59DaysPastDueNotWorse': 'times_30_59_days_late',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines',
    'NumberOfTimes90DaysLate': 'times_90_days_late',
    'NumberRealEstateLoansOrLines': 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse': 'times_60_89_days_late',
    'NumberOfDependents': 'dependents',
    'SeriousDlqin2yrs': 'target',
}
df = df.rename(columns=COLUMN_MAP)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.dtypes

## 2. Class Distribution

The target `SeriousDlqin2yrs` indicates whether a borrower experienced 90+ day delinquency within 2 years. A strongly imbalanced dataset is expected for credit default prediction.

In [ ]:
class_counts = df['target'].value_counts()
class_pct = df['target'].value_counts(normalize=True) * 100

print('Class distribution:')
for cls, count in class_counts.items():
    label = 'Default' if cls == 1 else 'No Default'
    print(f'  {label} ({cls}): {count:,} ({class_pct[cls]:.2f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No Default (0)', 'Default (1)'], class_counts.values,
       color=['#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='white')
ax.set_title('Target Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# ~6.7% positive class — significant imbalance requiring scale_pos_weight in XGBoost

## 3. Missing Values

Two features are expected to have missing values: `monthly_income` and `dependents`. Both will be imputed with the median in the preprocessing pipeline.

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

print('Features with missing values:')
print(missing_df.to_string())

## 4. Feature Distributions

We look at distributions of all 10 features to understand scale, skewness, and the presence of extreme outliers. Highly skewed features like `revolving_utilization` and `debt_ratio` will be capped at the 99th percentile before scaling.

In [ ]:
feature_cols = [
    'revolving_utilization', 'age', 'times_30_59_days_late', 'debt_ratio',
    'monthly_income', 'open_credit_lines', 'times_90_days_late',
    'real_estate_loans', 'times_60_89_days_late', 'dependents'
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    data = df[col].dropna()
    # Cap display at 99th pct to make histograms readable
    cap = data.quantile(0.99)
    axes[i].hist(data.clip(upper=cap), bins=50, color='steelblue', alpha=0.75, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=7)

fig.suptitle('Feature Distributions (capped at 99th pct for display)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
df[feature_cols].describe().T.round(2)

### Outlier Observations

Several features have extreme outliers:
- `revolving_utilization`: max values > 50 (physically possible but extreme)
- `debt_ratio`: max values in the thousands
- `monthly_income`: extremely high earners

These will be capped at the 99th percentile in preprocessing to prevent them from dominating the model.

In [ ]:
print('99th percentile values:')
for col in feature_cols:
    p99 = df[col].quantile(0.99)
    pmax = df[col].max()
    ratio = pmax / p99 if p99 > 0 else float('inf')
    flag = ' <-- extreme outliers' if ratio > 10 else ''
    print(f'  {col:<35} p99={p99:>10.2f}  max={pmax:>12.2f}{flag}')

## 5. Correlation Analysis

In [ ]:
corr = df[feature_cols + ['target']].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    xticklabels=[c.replace('_', '\n') for c in corr.columns],
    yticklabels=[c.replace('_', '\n') for c in corr.columns],
    annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with target (sorted):')
print(corr['target'].drop('target').sort_values(ascending=False).round(3).to_string())

## Summary

| Observation | Finding | Preprocessing Action |
|---|---|---|
| Class imbalance | 6.68% positive class | `scale_pos_weight=13.96` in XGBoost |
| Missing values | `monthly_income` (29k), `dependents` (3.9k) | Median imputation |
| Extreme outliers | `revolving_utilization`, `debt_ratio`, `monthly_income` | Winsorize at 99th pct |
| Feature scales | Wide variation across features | StandardScaler |
| Strongest predictor | `revolving_utilization` has highest correlation with target | Confirmed by SHAP in notebook 04 |

Proceed to `02_preprocessing.ipynb` to build and validate the preprocessing pipeline.